# 03. Modeling – Stroke Prediction

Notebook này thực hiện quá trình xây dựng, đánh giá và lựa chọn mô hình dự đoán đột quỵ.

Quy trình chính:
- Chuẩn bị dữ liệu train/test
- Đánh giá 6 mô hình bằng Stratified 5-Fold Cross-Validation
- Sử dụng SMOTE bên trong Pipeline để xử lý mất cân bằng dữ liệu
- Lựa chọn các mô hình dựa trên kết quả Cross-Validation
- Tuning Random Forest và XGBoost
- Lựa chọn mô hình cuối cùng
- Xác định Decision Threshold
- Đánh giá mô hình trên Test Set

## 1. Import thư viện

Các thư viện được sử dụng cho quá trình xây dựng, đánh giá và tuning mô hình.

In [19]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from src.models import get_models
from src.tuning import get_tuning_models, get_tuning_params
from src.preprocessing import RANDOM_STATE, get_preprocessor, load_data, split_data

print("Import OK")

Import OK


### Nhận xét

Các thư viện cần thiết cho quá trình xây dựng và đánh giá mô hình đã được import thành công.

Trong quá trình modeling, `Pipeline` được sử dụng để kết hợp tiền xử lý dữ liệu, SMOTE và mô hình. Điều này giúp hạn chế Data Leakage trong quá trình Cross-Validation.

## 2. Chuẩn bị dữ liệu

Dữ liệu được load và chia thành tập huấn luyện (Training Set) và tập kiểm tra (Test Set).

Tập Test được giữ riêng và chỉ sử dụng ở bước đánh giá cuối cùng.

In [20]:
df = load_data()

X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4088, 10)
X_test: (1022, 10)
y_train: (4088,)
y_test: (1022,)


### Nhận xét

Dữ liệu được chia thành:

- Training Set: 4.088 mẫu
- Test Set: 1.022 mẫu
- Số lượng biến đầu vào: 10

Training Set được sử dụng cho Cross-Validation và tuning mô hình. Test Set được giữ riêng để đánh giá mô hình cuối cùng.

## 3. Thiết lập Cross-Validation

Sử dụng Stratified 5-Fold Cross-Validation để đánh giá mô hình trên Training Set.

Stratified K-Fold giúp duy trì tỷ lệ giữa hai lớp của biến mục tiêu trong các fold.

In [21]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = [
    "recall",
    "precision",
    "f1",
    "roc_auc",
    "average_precision"
]

print("Cross-Validation: 5-Fold Stratified")
print("Metrics:", scoring)

Cross-Validation: 5-Fold Stratified
Metrics: ['recall', 'precision', 'f1', 'roc_auc', 'average_precision']


### Nhận xét

Mỗi mô hình được đánh giá bằng 5-fold Cross-Validation với các chỉ số:

- Recall
- Precision
- F1-score
- ROC-AUC
- Average Precision

Trong bài toán dự đoán đột quỵ, dữ liệu mất cân bằng nên Average Precision và Recall được quan tâm bên cạnh các chỉ số còn lại.

## 4. Đánh giá 6 mô hình bằng Cross-Validation

Các mô hình được đánh giá gồm:

1. Logistic Regression
2. Decision Tree
3. KNN
4. SVM
5. Random Forest
6. XGBoost

Mỗi mô hình được đặt trong Pipeline gồm:

**Preprocessing → SMOTE → Model**

SMOTE được thực hiện bên trong Pipeline để tránh Data Leakage giữa các fold.

In [22]:
rows = []

for name, model in get_models().items():
    pipe = Pipeline([
        ("preprocessor", get_preprocessor()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

    res = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    row = {"model": name}

    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()

    rows.append(row)

    print("Xong:", name)

cv_results = pd.DataFrame(rows).round(3)

cv_results

Xong: Logistic Regression
Xong: Decision Tree
Xong: KNN
Xong: SVM
Xong: Random Forest
Xong: XGBoost


,model,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Logistic Regression,0.754,0.039,0.131,0.009,0.223,0.015,0.836,0.011,0.192,0.036
1,Decision Tree,0.221,0.040,0.120,0.020,0.155,0.026,0.569,0.019,0.065,0.007
2,KNN,0.371,0.067,0.113,0.022,0.173,0.033,0.671,0.026,0.088,0.011
3,SVM,0.458,0.040,0.116,0.008,0.185,0.012,0.779,0.018,0.137,0.030
4,Random Forest,0.085,0.025,0.122,0.034,0.100,0.028,0.788,0.012,0.123,0.005
5,XGBoost,0.106,0.037,0.168,0.085,0.129,0.053,0.799,0.009,0.143,0.012


### Nhận xét

Kết quả Cross-Validation cho thấy các mô hình có hiệu quả khác nhau trên dữ liệu mất cân bằng.

Logistic Regression đạt Average Precision cao nhất trong 6 mô hình với giá trị 0.192, đồng thời có ROC-AUC = 0.836 và Recall = 0.754.

XGBoost đạt Average Precision = 0.143 và ROC-AUC = 0.799.

Các kết quả này được sử dụng làm cơ sở để lựa chọn mô hình cho các bước tiếp theo.

In [23]:
cv_results.to_csv("../results/cv_results.csv", index=False)

print("Đã lưu kết quả vào: ../results/cv_results.csv")

Đã lưu kết quả vào: ../results/cv_results.csv


### Nhận xét

Kết quả Cross-Validation được lưu vào `results/cv_results.csv` để sử dụng cho báo cáo và các bước tiếp theo.

## 5. Lựa chọn 2 mô hình từ kết quả Cross-Validation

Dựa trên kết quả Cross-Validation, Average Precision được sử dụng làm tiêu chí chính để lựa chọn 2 mô hình.

Average Precision phù hợp với bài toán dữ liệu mất cân bằng vì đánh giá chất lượng dự đoán đối với lớp dương thông qua Precision-Recall.

In [24]:
# Đọc lại kết quả CV đã lưu
cv_results = pd.read_csv("../results/cv_results.csv")

# Chọn 2 mô hình tốt nhất theo Average Precision
top_2_models = (
    cv_results
    .sort_values("average_precision_mean", ascending=False)
    .head(2)
)

top_2_models[[
    "model",
    "average_precision_mean",
    "roc_auc_mean",
    "f1_mean",
    "recall_mean",
    "precision_mean"
]]

,model,average_precision_mean,roc_auc_mean,f1_mean,recall_mean,precision_mean
0,Logistic Regression,0.192,0.836,0.223,0.754,0.131
5,XGBoost,0.143,0.799,0.129,0.106,0.168


In [25]:
print("2 mô hình được chọn theo Average Precision:")
for _, row in top_2_models.iterrows():
    print(f"- {row['model']}: AP = {row['average_precision_mean']:.3f}")

2 mô hình được chọn theo Average Precision:
- Logistic Regression: AP = 0.192
- XGBoost: AP = 0.143


### Kết quả lựa chọn

Hai mô hình có Average Precision cao nhất là:

1. Logistic Regression — AP = 0.192
2. XGBoost — AP = 0.143

Đây là kết quả lựa chọn dựa trên Cross-Validation, không phải kết quả đánh giá trên Test Set.

## 6. Hyperparameter Tuning

Sau khi đánh giá các mô hình bằng Cross-Validation, bước tiếp theo là tối ưu siêu tham số cho Random Forest và XGBoost.

Quá trình tuning sử dụng RandomizedSearchCV với:
- Stratified 5-Fold Cross-Validation
- SMOTE được đặt bên trong Pipeline
- Average Precision làm tiêu chí đánh giá
- Chỉ sử dụng Training Set, không sử dụng Test Set

### 6.1. Tuning Random Forest

Random Forest được tối ưu các siêu tham số như số lượng cây, độ sâu cây, số mẫu tối thiểu để tách nhánh, số mẫu tối thiểu tại lá và số lượng đặc trưng được sử dụng tại mỗi lần chia.

In [26]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

from src.tuning import get_tuning_models, get_tuning_params

In [27]:
rf_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["Random Forest"]),
])

print("Random Forest pipeline OK")

Random Forest pipeline OK


In [28]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=get_tuning_params()["Random Forest"],
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("Random Forest RandomizedSearchCV OK")

Random Forest RandomizedSearchCV OK


In [29]:
rf_search.fit(X_train, y_train)

print("Random Forest tuning hoàn tất.")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forest tuning hoàn tất.


In [30]:
print("Best CV Average Precision:", rf_search.best_score_)

Best CV Average Precision: 0.15630700467257658


In [31]:
print("Best Parameters:")
print(rf_search.best_params_)

Best Parameters:
{'model__max_depth': 5, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__n_estimators': 290}


### Nhận xét

Random Forest đã được tối ưu bằng RandomizedSearchCV với 20 bộ tham số và 5-fold Cross-Validation.

Bộ tham số tốt nhất được xác định gồm:
- `n_estimators = 290`
- `max_depth = 5`
- `max_features = log2`
- `min_samples_split = 6`
- `min_samples_leaf = 2`

Best CV Average Precision của Random Forest được sử dụng để so sánh với XGBoost trong bước tuning tiếp theo.

### 6.2. Tuning XGBoost

XGBoost được tối ưu bằng RandomizedSearchCV với cùng quy trình Cross-Validation như Random Forest.

Pipeline gồm:
- Tiền xử lý dữ liệu
- SMOTE để xử lý mất cân bằng lớp
- Mô hình XGBoost

Average Precision được sử dụng làm tiêu chí đánh giá trong quá trình tuning.

In [43]:
xgb_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["XGBoost"]),
])

print("XGBoost pipeline OK")

XGBoost pipeline OK


In [44]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=get_tuning_params()["XGBoost"],
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("XGBoost RandomizedSearchCV OK")

XGBoost RandomizedSearchCV OK


In [45]:
xgb_search.fit(X_train, y_train)

print("XGBoost tuning hoàn tất.")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
XGBoost tuning hoàn tất.


In [46]:
print("Best CV Average Precision:", xgb_search.best_score_)

Best CV Average Precision: 0.16063964880132103


In [47]:
print("Best Parameters:")
print(xgb_search.best_params_)

Best Parameters:
{'model__colsample_bytree': 0.7, 'model__gamma': 0.1, 'model__learning_rate': 0.03, 'model__max_depth': 2, 'model__min_child_weight': 4, 'model__n_estimators': 234, 'model__subsample': 0.7}


### Nhận xét

XGBoost được tối ưu bằng RandomizedSearchCV với 20 bộ tham số và 5-fold Cross-Validation.

Bộ tham số tốt nhất được xác định gồm:
- `n_estimators = 234`
- `max_depth = 2`
- `learning_rate = 0.03`
- `subsample = 0.7`
- `colsample_bytree = 0.7`
- `min_child_weight = 4`
- `gamma = 0.1`

Best CV Average Precision của XGBoost đạt **0.1606**. Kết quả này được sử dụng để so sánh với Random Forest trong bước lựa chọn mô hình cuối cùng.

In [48]:
tuning_results = pd.DataFrame({
    "model": ["Random Forest", "XGBoost"],
    "best_cv_average_precision": [
        rf_search.best_score_,
        xgb_search.best_score_
    ]
})

tuning_results

,model,best_cv_average_precision
0,Random Forest,0.156307
1,XGBoost,0.160640


### Nhận xét

Kết quả tuning cho thấy:

- Random Forest đạt Best CV Average Precision = **0.1563**.
- XGBoost đạt Best CV Average Precision = **0.1606**.

Trong bước tuning, XGBoost có Average Precision trên Cross-Validation cao hơn Random Forest. Vì vậy, XGBoost được chọn làm mô hình cuối để thực hiện các bước tiếp theo.

Mô hình chỉ được đánh giá trên tập test ở bước cuối, sau khi hoàn tất quá trình lựa chọn và tuning.

In [49]:
tuning_results.to_csv("../results/tuning_results.csv", index=False)

print("Đã lưu results/tuning_results.csv")

Đã lưu results/tuning_results.csv


In [50]:
final_model = xgb_search.best_estimator_

print("Final model: XGBoost")

Final model: XGBoost


In [51]:
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF probabilities:", oof_proba.shape)

OOF probabilities: (4088,)


In [52]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.10, 0.91, 0.01)

f1_scores = []

for threshold in thresholds:
    oof_pred = (oof_proba >= threshold).astype(int)
    f1_scores.append(f1_score(y_train, oof_pred))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print("Best threshold:", best_threshold)
print("Best CV F1:", best_f1)

Best threshold: 0.5699999999999997
Best CV F1: 0.22950819672131148


### Nhận xét

Threshold được xác định trên tập train bằng dự đoán Out-of-Fold (OOF), không sử dụng tập test.

Kết quả cho thấy:
- Best threshold = **0.57**
- Best CV F1 = **0.2295**

Threshold **0.57** sẽ được sử dụng để chuyển xác suất dự đoán của XGBoost thành nhãn 0/1 trong bước đánh giá cuối cùng trên tập test.

In [53]:
final_model.fit(X_train, y_train)

print("Final model đã được fit trên toàn bộ tập train.")

Final model đã được fit trên toàn bộ tập train.


In [54]:
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.57).astype(int)

print("Test prediction hoàn tất.")

Test prediction hoàn tất.


In [55]:
test_results = pd.DataFrame({
    "model": ["XGBoost"],
    "threshold": [0.57],
    "recall": [recall_score(y_test, y_test_pred)],
    "precision": [precision_score(y_test, y_test_pred)],
    "f1": [f1_score(y_test, y_test_pred)],
    "roc_auc": [roc_auc_score(y_test, y_test_proba)],
    "average_precision": [average_precision_score(y_test, y_test_proba)]
})

test_results.round(4)

,model,threshold,recall,precision,f1,roc_auc,average_precision
0,XGBoost,0.57,0.68,0.156,0.2537,0.8355,0.2606


In [56]:
test_results.to_csv("../results/test_results.csv", index=False)

print("Đã lưu results/test_results.csv")

Đã lưu results/test_results.csv


### Nhận xét

Mô hình XGBoost sau khi tuning được đánh giá trên tập test với threshold = **0.57**.

Kết quả thu được:
- Recall = **0.6800**
- Precision = **0.1560**
- F1-score = **0.2537**
- ROC-AUC = **0.8355**
- Average Precision = **0.2606**

Mô hình đạt ROC-AUC = 0.8355 và Average Precision = 0.2606 trên tập test. Recall đạt 0.68, cho thấy mô hình phát hiện được phần lớn các trường hợp thuộc lớp dương trong tập test.

Threshold 0.57 được xác định trước bằng dự đoán Out-of-Fold trên tập train và được giữ cố định khi đánh giá trên tập test.

In [57]:
import os

print("cv_results.csv:", os.path.exists("../results/cv_results.csv"))
print("tuning_results.csv:", os.path.exists("../results/tuning_results.csv"))
print("test_results.csv:", os.path.exists("../results/test_results.csv"))

cv_results.csv: True
tuning_results.csv: True
test_results.csv: True
